# What did this layout actually cost

`CalibrationMapper` picks physical qubits from calibration data. The selection receipt records that
choice so somebody else can check it later.

Until 0.12.0 the receipt always described the layout the mapper **recommended**, whatever the caller
ran. Override the layout, which is a normal thing to do, and you got a clean looking receipt for a
run that never happened. This notebook shows the fix, on a real IBM Fez calibration snapshot from
the repository.

In [1]:
import warnings

# requests warns at import about its vendored urllib3 version and prints an absolute path with it.
# Nothing to do with this notebook, and it lands in the middle of the output below.
warnings.filterwarnings("ignore", message=r"urllib3 \(.*\) or chardet")

from qb_compiler.calibration.static_provider import StaticCalibrationProvider
from qb_compiler.ir.circuit import QBCircuit
from qb_compiler.ir.operations import QBGate
from qb_compiler.passes.mapping import CalibrationMapper, selection_receipt

provider = StaticCalibrationProvider.from_json(
    "../tests/fixtures/calibration_snapshots/ibm_fez_2026_03_14.json"
)
props = provider.properties
print(f"{props.backend}, {props.n_qubits} qubits, snapshot taken {props.timestamp}")

circuit = QBCircuit(n_qubits=5, name="ghz5")
circuit.add_gate(QBGate(name="h", qubits=(0,)))
for control, target in [(0, 1), (1, 2), (2, 3), (3, 4)]:
    circuit.add_gate(QBGate(name="cx", qubits=(control, target)))

mapper = CalibrationMapper(props)
result = mapper.run(circuit, {})
recommended = result.metadata["initial_layout"]
print("recommended layout:", recommended)
print(f"score             : {result.metadata['calibration_score']:.6f}  (lower is better)")

ibm_fez, 156 qubits, snapshot taken 2026-03-14T08:41:35.899941+00:00
recommended layout: {1: 23, 2: 22, 0: 16, 3: 21, 4: 20}
score             : 0.602721  (lower is better)


The score is a sum of weighted penalties on this one snapshot. It ranks layouts against each other
on this snapshot and means nothing on its own. It is not a fidelity.

In [2]:
import json

receipt = selection_receipt(result, calibration=props)
print(json.dumps(receipt, indent=2, default=str))

{
  "schema": "qb.selection_receipt.v1",
  "objective": "calibration-aware layout (CalibrationMapper: gate error + coherence + readout + T1 asymmetry + temporal correlation, VF2 subgraph search)",
  "selected_layout": {
    "1": 23,
    "2": 22,
    "0": 16,
    "3": 21,
    "4": 20
  },
  "selected_score": 0.6027205467704855,
  "score_breakdown": {
    "gate_error": 0.1188336855964986,
    "coherence": 0.17341815653934703,
    "readout": 0.223388671875,
    "t1_asymmetry": 0.08708003275963991,
    "correlation": 0.0,
    "total": 0.6027205467704856
  },
  "describes_executed_layout": true,
  "calibration_hash": "6c75eb81d3e7db62",
  "calibration_freshness": {
    "calibration_timestamp": "2026-03-14T08:41:35.899941+00:00",
    "age_minutes": 229718.93,
    "tolerance_minutes": 30.0,
    "tolerance_basis": "builtin_default_not_measured_for_this_device",
    "exceeds_default_tolerance": true,
    "timestamp_status": "measured",
    "note": "Age is measured, tolerance is a fixed default.

## Now run something else

A campaign here executed `[6, 5, 4, 3, 2]` and got a receipt naming `{144, 143, 136, 123, 124}`.
Both numbers were internally consistent. One of them described a circuit that never ran.

Pass `executed_layout=` and a scorer, and the receipt describes what ran.

In [3]:
import functools

# Deliberately run a worse region than the mapper picked.
candidates = mapper.rank_layouts(circuit, top_k=6, max_overlap=0.0)
executed = candidates[-1].layout
print("executed instead:", executed)

honest = selection_receipt(
    result,
    calibration=props,
    executed_layout=executed,
    scorer=functools.partial(mapper.score_layout, circuit=circuit),
)

print()
print("selected_layout   :", honest["selected_layout"])
print("recommended_layout:", honest["recommended_layout"])
print(f"selected_score    : {honest['selected_score']:.6f}")
print(f"recommended_score : {honest['recommended_score']:.6f}")
print(f"penalty           : {honest['score_penalty_vs_recommended']:+.6f}")
print()
print(honest["divergence_note"])

executed instead: {1: 143, 2: 136, 0: 144, 3: 123, 4: 122}

selected_layout   : {'1': 143, '2': 136, '0': 144, '3': 123, '4': 122}
recommended_layout: {'1': 23, '2': 22, '0': 16, '3': 21, '4': 20}
selected_score    : 0.620688
recommended_score : 0.602721
penalty           : +0.017967

OVERRIDDEN: the layout that ran is not the one the pass recommended. 5 of 5 logical qubits map elsewhere; the calibration score of what ran is +0.017967 against the recommendation.


`describes_executed_layout` is on every receipt now, including ones where you passed nothing,
where it is `True` because the recommendation is what ran. A reader never has to infer it.

The per signal breakdown is dropped when a different layout ran, with a note saying why: the
breakdown belongs to the recommendation, and leaving it beside another layout is how this went
wrong in the first place.

In [4]:
print("no executed_layout passed:", receipt["describes_executed_layout"])
print("executed_layout passed   :", honest["describes_executed_layout"])
print("matches recommendation   :", honest["executed_layout_matches_recommendation"])
print("breakdown                :", honest["score_breakdown"])
print("why                      :", honest["score_breakdown_note"])

no executed_layout passed: True
executed_layout passed   : True
matches recommendation   : False
breakdown                : {}
why                      : the per signal breakdown is computed for the recommendation only, and is omitted here because a different layout ran


## How old was the data behind the choice

A compiler using hour old calibration and one using fresh calibration produce identical output.
Every receipt now carries the age.

In [5]:
for key, value in receipt["calibration_freshness"].items():
    print(f"{key:26s} {value}")

calibration_timestamp      2026-03-14T08:41:35.899941+00:00
age_minutes                229718.93
tolerance_minutes          30.0
tolerance_basis            builtin_default_not_measured_for_this_device
exceeds_default_tolerance  True
timestamp_status           measured
note                       Age is measured, tolerance is a fixed default. A tolerance measured for this specific backend, and the publication delay that makes the reported age an over-estimate, are not derivable from this package.


That snapshot is a file in the repository, so of course it is old. The point is that the number is
there at all, and that `tolerance_basis` says in words that the 30 minute tolerance is a builtin
default rather than a measurement of this device. A default presented as a measurement would be
worse than no number.

## The alternatives, and why two of them were the same option

`rank_layouts` returns the trade space behind the winner. Raw, the top of the ranking is not as
diverse as it looks.

In [6]:
raw = mapper.rank_layouts(circuit, top_k=6, diversify=False)
for candidate in raw:
    print(f"rank {candidate.rank}  score {candidate.score:.6f}  qubits {candidate.physical_qubits}")

rank 0  score 0.602721  qubits (16, 20, 21, 22, 23)
rank 1  score 0.602721  qubits (16, 20, 21, 22, 23)
rank 2  score 0.620688  qubits (122, 123, 136, 143, 144)
rank 3  score 0.620688  qubits (122, 123, 136, 143, 144)
rank 4  score 0.637860  qubits (123, 124, 136, 143, 144)
rank 5  score 0.637860  qubits (123, 124, 136, 143, 144)


Ranks that share a physical qubit set are the same hardware with the logical labels permuted: one
option presented as several. `max_overlap` compares the sets directly and removes them.

In [7]:
distinct = mapper.rank_layouts(circuit, top_k=6, diversify=False, max_overlap=0.0)
print(f"{len(raw)} candidates in, {len(distinct)} genuinely different regions out")
for candidate in distinct:
    print(f"  score {candidate.score:.6f}  qubits {candidate.physical_qubits}")

6 candidates in, 2 genuinely different regions out
  score 0.602721  qubits (16, 20, 21, 22, 23)
  score 0.620688  qubits (122, 123, 136, 143, 144)


## Against Qiskit, scored honestly

Qiskit's own `VF2PostLayout` does calibration aware layout when it is given a Target with error
rates. Build one from the same snapshot and see where it lands.

In [8]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Measure, Parameter
from qiskit.circuit.library import CZGate, RZGate, SXGate, XGate
from qiskit.transpiler import InstructionProperties, Target

target = Target(num_qubits=props.n_qubits)
readout = {q.qubit_id: (q.readout_error or 0.01) for q in props.qubit_properties}
single = {(q,): InstructionProperties(error=1e-4, duration=32e-9) for q in range(props.n_qubits)}
target.add_instruction(SXGate(), single)
target.add_instruction(XGate(), dict(single))
target.add_instruction(
    RZGate(Parameter("theta")),
    {(q,): InstructionProperties(error=0.0, duration=0.0) for q in range(props.n_qubits)},
)
target.add_instruction(
    Measure(),
    {(q,): InstructionProperties(error=readout.get(q, 0.01), duration=1e-6) for q in range(props.n_qubits)},
)
target.add_instruction(
    CZGate(),
    {
        tuple(gp.qubits): InstructionProperties(
            error=float(gp.error_rate), duration=(gp.gate_time_ns or 68.0) * 1e-9
        )
        for gp in props.gate_properties
        if len(gp.qubits) == 2 and gp.error_rate is not None
    },
)

qc = QuantumCircuit(5)
qc.h(0)
for control, target_qubit in [(0, 1), (1, 2), (2, 3), (3, 4)]:
    qc.cx(control, target_qubit)
qc.measure_all()

transpiled = transpile(qc, target=target, optimization_level=3, seed_transpiler=11)
qiskit_layout = {i: p for i, p in enumerate(transpiled.layout.initial_index_layout(filter_ancillas=True))}

print("qiskit picked :", sorted(qiskit_layout.values()))
print("qb-compiler   :", sorted(recommended.values()))
print()
print(f"qiskit layout scored on our objective : {mapper.score_layout(qiskit_layout, circuit):.6f}")
print(f"our layout scored on our objective    : {result.metadata['calibration_score']:.6f}")

qiskit picked : [123, 124, 136, 143, 144]
qb-compiler   : [16, 20, 21, 22, 23]

qiskit layout scored on our objective : 0.637860
our layout scored on our objective    : 0.602721


Read that carefully. Both layouts are scored with **our** objective, so our pick wins by
construction. It is not evidence that our layout runs better on the device, and quoting it as one
would be dishonest.

What it does show is that the two tools land in different regions of the same chip from the same
data, and that Qiskit's choice is inside our own candidate list, a few ranks down.

The hardware comparisons we have run are in the README: on GHZ circuits the delta was between
+0.2 and +5.3 points of measured fidelity depending on the circuit and the calibration window, and
on a 16 qubit chemistry circuit the comparison against Sabre came out a **null**. Layout selection
is a commodity in this ecosystem. The receipt around it is not.

## What this bought

| | |
|---|---|
| free | the layout that ran, its score, the calibration fingerprint and age, the ranked alternatives |
| free | signing with your own key, and offline verification by anyone (notebook 28) |
| paid | keys you do not have to hold, countersigning, and receipts that outlive the machine that made them |
| never claimed | that the chosen layout is optimal, or that the score is a fidelity |